# Semana 03: Sensores Avançados, Atuadores e Mapeamento de Sinais no CLP

## Módulo de Controladores e Dispositivos — Fábrica Virtual Smart N1

Este notebook apresenta a interpretação de dispositivos avançados de aquisição (sensores fotoelétricos PSD, ultrassônicos Time-of-Flight e identificação RFID), controle de atuadores (solenoides, inversores VFD e servomotores via PWM) e o **mapeamento de sinais de campo na memória do CLP (Tabela de Imagem PII e Ciclo de SCAN)** sob a perspectiva de TI.

### Objetivos de aprendizagem
- Entender o mapeamento de hardware para registradores de memória do CLP (Tabela de Imagem de Entradas - PII).
- Analisar a mecânica do **Ciclo de SCAN** (Leitura de Entradas → Execução da Lógica → Atualização de Saídas → Comunicação) e seu impacto no tempo de reação do software.
- Interpretar sensores de proximidade fotoelétricos e ultrassônicos Time-of-Flight (ToF) gerando dados contínuos de distância.
- Compreender a leitura e escrita de dados estruturados em Tags RFID industriais (Payloads em bytes/hexadecimal).
- Controlar atuadores discretos e variáveis (Inversores VFD e Servomotores via sinal PWM e rampas de aceleração).
- Executar simuladores em Python para a Tabela PII, leitura de memória RFID e ciclo de SCAN.

---


## 1. Fundamentação teórica

### 1.1 Mapeamento de Memória no CLP: Tabela PII e Ciclo de SCAN

![Mapeamento de Sinais na Tabela de Imagem de Entrada PII](img/sensores_sinais_ti.jpg)

Em um controlador industrial (CLP/PC Industrial), os programas de software não leem os pinos físicos diretamente durante a execução da lógica. Existe uma camada de abstração na memória de TI:

1. **Tabela de Imagem de Entrada (Process Image Input - PII):** No início do ciclo, o firmware lê todos os pinos de hardware físicos (ex: `%I0.0` a `%I0.7`) e atualiza uma tabela de bytes na RAM do controlador.
2. **Execução do Programa:** A lógica de software lê os valores armazenados na PII, garantindo determinismo (os valores das entradas não mudam no meio da execução do código).
3. **Tabela de Imagem de Saída (Process Image Output - PIO):** As variáveis de saída alteradas pelo programa (ex: `%Q0.0` a `%Q0.7`) são gravadas na memória RAM.
4. **Atualização Física das Saídas:** Ao final do ciclo, o firmware descarrega a PIO nos cartões físicos conectando relés e acionadores.
5. **SCAN Time:** Tempo total para completar 1 ciclo. Se o SCAN Time for $10\text{ ms}$, o controlador lê e atualiza dados a $100\text{ Hz}$.


### 1.2 Sensores Avançados e Transmissão de Dados

- **Sensores Ultrassônicos (Time-of-Flight - ToF):** Medem o tempo decorrido entre a emissão de um pulso sonoro e seu eco refletido pelo objeto ($d = \frac{v \cdot t}{2}$). Convertem a distância medida em um sinal analógico contínuo de $4\text{ a }20\text{mA}$ ou variável `float` em milímetros.
- **Identificação por Radiofrequência (RFID Industrial):** Leitura sem contato de chips fixados em pallets ou peças. O leitor captura payloads estruturados contendo UID do chip, código do produto e lote de produção em pacotes de bytes em formato Hexadecimal (`0x4A8B...`).


### 1.3 Atuadores e Sinal de Comando (Discreto vs PWM)

- **Válvulas Solenoide e Cilindros Pneumáticos:** Recebem sinal elétrico discreto ($0\text{V}$ desativada / $+24\text{V}$ ativada) para avanço/recuo mecânico.
- **Inversores de Frequência (VFD) e Servomotores:** Controlam a velocidade de rotação e torque de motores AC/DC. O sinal de velocidade enviado pelo software de TI pode ser:
  - Sinal Analógico ($0\text{ a }10\text{V}$);
  - **Sinal PWM:** Onde a variação do Duty Cycle de $0\%$ a $100\%$ comanda a velocidade da linha de $0\text{ RPM}$ a $1800\text{ RPM}$;
  - Telegramas digitais via redes industriais (Profinet, Ethernet/IP, Modbus TCP).

---


## 2. Arquitetura da atividade

**Hardware de Campo (Sensores / RFID) → Leitura PII na RAM → Ciclo de SCAN (Software) → Comando PWM/PIO → Atuadores**

---


## 3. Prática — Simulador de Ciclo de SCAN do CLP e Leitura RFID em Python

### Passo 1 — Simulador da Tabela de Imagem (PII) e Ciclo de SCAN
Execute o script em Python para simular a leitura de entradas físicas, o SCAN Time e a atualização da Tabela PII.

In [ ]:
import time
import random

class SimulaçãoCLP:
    def __init__(self):
        # Tabela PII (8 Entradas Digitais: %I0.0 a %I0.7)
        self.PII = [False] * 8
        # Tabela PIO (8 Saídas Digitais: %Q0.0 a %Q0.7)
        self.PIO = [False] * 8
        self.registrador_pwm_velocidade = 0.0 # Duty Cycle 0-100%
        
    def ler_entradas_fisicas(self):
        # Simula estado dos fios nos bornes do CLP
        self.PII[0] = random.choice([True, False]) # Sensor Presenca Peca (%I0.0)
        self.PII[1] = True                        # Botao Emergencia NF (%I0.1)
        self.PII[2] = random.choice([True, False]) # Sensor Temperatura Alta (%I0.2)
        
    def executar_logica_software(self):
        # Se peca presente (%I0.0) E emergencia OK (%I0.1), liga esteira (%Q0.0)
        if self.PII[0] and self.PII[1]:
            self.PIO[0] = True
            self.registrador_pwm_velocidade = 75.0 # Setpoint PWM 75%
        else:
            self.PIO[0] = False
            self.registrador_pwm_velocidade = 0.0
            
    def ciclo_de_scan(self):
        inicio = time.time()
        self.ler_entradas_fisicas()
        self.executar_logica_software()
        scan_time_ms = (time.time() - inicio) * 1000 + random.uniform(1.2, 3.5)
        return scan_time_ms

clp = SimulaçãoCLP()
scan_ms = clp.ciclo_de_scan()
print("=== SIMULAÇÃO DO CICLO DE SCAN DO CLP ===")
print(f"Tabela PII (Entradas) : {clp.PII}")
print(f"Tabela PIO (Saídas)   : {clp.PIO}")
print(f"Comando PWM Atuador   : {clp.registrador_pwm_velocidade}% Duty Cycle")
print(f"SCAN Time Medido      : {scan_ms:.2f} ms")


### Passo 2 — Leitor de Memory Payload de Tag RFID Industrial
Rode a célula para simular a leitura e decodificação do bloco de memória hexadecimal lido de uma Tag RFID.

In [ ]:
def decodificar_tag_rfid(payload_hex):
    # Transforma bloco de bytes hex em chaves estruturadas de software
    uid = payload_hex[:8]
    codigo_produto = bytes.fromhex(payload_hex[8:24]).decode('utf-8', errors='ignore')
    lote = int(payload_hex[24:28], 16)
    return {
        "uid_chip": f"0x{uid}",
        "produto": codigo_produto.strip(),
        "numero_lote": lote
    }

# Exemplo de payload lido pelo leitor RFID
hex_bytes = "4A8B9C1D534D4152545F4E3107E0" # Contem 'SMART_N1' + Lote 2016
tag_dados = decodificar_tag_rfid(hex_bytes)

print("=== DECODIFICAÇÃO DE PAYLOAD DA TAG RFID ===")
print(json.dumps(tag_dados, indent=2))


---

## 4. Exercícios de fixação e avaliação

### Questão 1
O que é a Tabela de Imagem de Entrada (PII) no CLP e por que o software lê a memória RAM em vez de acessar diretamente os pinos elétricos durante a execução do programa?

### Questão 2
Como o tempo de ciclo (SCAN Time) do controlador afeta a capacidade do software de detectar pulsos rápidos de sensores discretos em esteiras de alta velocidade?

### Questão 3
De que maneira o controle por sinal PWM permite ao controlador ajustar a velocidade de um atuador elétrico (ex: motor de esteira) de forma progressiva em vez de apenas liga/desliga?
